In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# --- 1. Carga de Datos ---
file_path = 'unificado (1).csv'
try:
    df = pd.read_csv(file_path, low_memory=False)
except FileNotFoundError:
    print(f"Error: El archivo '{file_path}' no fue encontrado. Asegúrate de que esté en la ubicación correcta.")
    exit()
except Exception as e:
    print(f"Error al cargar el archivo CSV: {e}")
    exit()

print("--- NOMBRES DE COLUMNAS DESPUÉS DE CARGAR EL CSV ---")
print(df.columns.tolist())
print("----------------------------------------------------")
print("\nDimensiones originales del DataFrame:", df.shape)

# --- 2. Preprocesamiento Consistente ---
df.replace('-', np.nan, inplace=True)


print("\nConvirtiendo columnas a tipo numérico (errores serán NaN)...")
for col in df.columns:
    if col in df:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("\nImputando valores faltantes con la mediana para cada columna...")

imputer = SimpleImputer(strategy='median')
df_imputed = imputer.fit_transform(df)
df = pd.DataFrame(df_imputed, columns=df.columns)


print(f"\nNúmero total de NaNs después de la imputación: {df.isnull().sum().sum()}")
if df.isnull().sum().sum() > 0:
    print("Advertencia: Aún quedan NaNs después de la imputación. Revisa las columnas:")
    print(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print("No quedan NaNs en el DataFrame después de la imputación.")

print("\nPrimeras filas del DataFrame preprocesado:")
print(df.head())

# --- 3. Definición de Características (X) y Variables Objetivo (y) ---

target_columns = [
    'ESCHOM',  # Escolaridad del hombre
    'ESCMUJ',  # Escolaridad de la mujer
    'CIUOHOM',  # Ocupación del hombre (asumiendo que esto es CIUOHOM)
    'CIUOMUJ'   # Ocupación de la mujer (asumiendo que esto es CIUOMUJ)
]

print(f"\nIntentando usar las siguientes columnas como variables objetivo: {target_columns}")
print("Columnas disponibles en el DataFrame en este punto:")
current_columns_list = df.columns.tolist()
print(current_columns_list)

# Verificar que todas las columnas objetivo existan
missing_targets = [col for col in target_columns if col not in df.columns]
if missing_targets:
    print(f"\nError DEFINITIVO: Las siguientes columnas objetivo NO se encuentran en el DataFrame: {missing_targets}")
    print("Revisa la lista de columnas disponibles impresas arriba y compara con los nombres que estás usando.")
    print("Posibles problemas: error de ortografía, mayúsculas/minúsculas, espacios extra, o las columnas realmente no existen en tu CSV con esos nombres.")
    exit()
else:
    print("\nTodas las columnas objetivo especificadas se encontraron en el DataFrame.")

y = df[target_columns]
X = df.drop(columns=target_columns) # Usar 'columns=' para múltiples columnas es más explícito

print(f"\nVariables objetivo (y) definidas con {len(target_columns)} columnas.")
print("Forma de X (características):", X.shape)
print("Forma de y (objetivo):", y.shape)

# --- 4. División en Conjuntos de Entrenamiento y Prueba ---

try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
except Exception as e:
    print(f"Error durante train_test_split: {e}")
    print("Esto puede ocurrir si X o y están vacíos o tienen dimensiones incompatibles.")
    exit()

print("\n--- Resultados de la División ---")
print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de X_test: {X_test.shape}")
print(f"Forma de y_train (objetivos de entrenamiento): {y_train.shape}")
print(f"Forma de y_test (objetivos de prueba): {y_test.shape}")

total_datos = len(df)
porcentaje_entrenamiento = len(X_train) / total_datos * 100
porcentaje_prueba = len(X_test) / total_datos * 100

print(f"\nPorcentaje de datos en el conjunto de entrenamiento: {porcentaje_entrenamiento:.2f}%")
print(f"Porcentaje de datos en el conjunto de prueba: {porcentaje_prueba:.2f}%")

print("\nPrimeras filas de X_train:")
print(X_train.head())
print("\nPrimeras filas de y_train (múltiples objetivos):")
print(y_train.head())

--- NOMBRES DE COLUMNAS DESPUÉS DE CARGAR EL CSV ---
['DEPREG', 'MUPREG', 'MESREG', 'AÑOREG', 'DIAOCU', 'MESOCU', 'AÑOOCU', 'DEPOCU', 'MUPOCU', 'EDADHOM', 'EDADMUJ', 'PPERHOM', 'PPERMUJ', 'NACHOM', 'NACMUJ', 'ESCHOM', 'ESCMUJ', 'CIUOHOM', 'CIUOMUJ']
----------------------------------------------------

Dimensiones originales del DataFrame: (81826, 19)

Convirtiendo columnas a tipo numérico (errores serán NaN)...

Imputando valores faltantes con la mediana para cada columna...

Número total de NaNs después de la imputación: 0
No quedan NaNs en el DataFrame después de la imputación.

Primeras filas del DataFrame preprocesado:
   DEPREG  MUPREG  MESREG  AÑOREG  DIAOCU  MESOCU  AÑOOCU  DEPOCU  MUPOCU  \
0     6.0   606.0    12.0  2020.0    19.0    11.0  2020.0     6.0   614.0   
1     7.0   704.0     2.0  2020.0     3.0     2.0  2020.0     7.0   704.0   
2     9.0   916.0     9.0  2020.0    21.0     9.0  2020.0     9.0   916.0   
3     8.0   801.0     3.0  2021.0    18.0    11.0  2020.0   

# New Section